# Cassini CDA Dust Analyzer: 3-OH Inference with Overlapping Batch Jobs (16b)

Same classification and outputs as `16_3oh_inference.ipynb`, but this notebook does not wait for each Vertex AI job before preparing the next batch:

1. **Submit (Step 6):** for each batch, the request file is written locally, uploaded and submitted as a batch job. The local file is then deleted and the next batch is prepared. So there is only one request file on disk at a time, about 1.4 GB.
2. **Collect (Step 7):** once all batches of this execution are submitted, the notebook waits for the jobs and downloads and parses each one as it finishes. It writes the same files as notebook 16.

**One execution = `BATCHES_PER_RUN` × `BATCH_SIZE` spectra** (default 10 × 1,000 = 10,000). Running the notebook again sends the next 10,000.

**Shared progress with notebook 16:** both notebooks use `inference_results_3oh/`.
- Spectra already processed by either notebook are skipped.
- Jobs submitted here are recorded in `submitted_jobs_16b.json`. If the kernel stops before they are collected, the next execution excludes their spectra from new batches and collects the jobs. Set `SUBMIT_NEW_BATCHES = False` to only collect.
- `17_3oh_progress_monitor.ipynb` shows the progress.

**Do not run notebooks 16 and 16b at the same time.** To switch, wait until notebook 16 prints "Batch done", interrupt it while it prepares the next request file, then delete `inference_results_3oh/batch_requests.jsonl`. This notebook refuses to start while that file exists.

## Step 1: Imports and Configuration

In [1]:
import os
import sys
import json
import time
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv

import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger("google_genai").setLevel(logging.WARNING)


# Repository root = the folder with pyproject.toml and the cda_dust_agent package, wherever the kernel was started.
# (Notebooks/cda_dust_agent/ is only a data folder, so the package's __init__.py is required.)
def is_repo_root(path):
    return (os.path.isfile(os.path.join(path, "pyproject.toml"))
            and os.path.isfile(os.path.join(path, "cda_dust_agent", "__init__.py")))

project_root = os.path.abspath(os.getcwd())
while not is_repo_root(project_root):
    parent = os.path.dirname(project_root)
    if parent == project_root:
        raise RuntimeError("Repository root (pyproject.toml + cda_dust_agent/) not found; start Jupyter inside the repository.")
    project_root = parent
study_dir = os.path.join(project_root, "Notebooks", "3oh_classification")
print(f"Repository root: {project_root}")

if not load_dotenv(dotenv_path=os.path.join(project_root, ".env")):
    print("Warning: no .env loaded; the defaults in cda_dust_agent/config.py are used for project and bucket.")
sys.path.extend([project_root, study_dir])

from cda_dust_agent.config import Config
import cda_3oh_common as c3

Repository root: /workspaces/cda-dust-analyzer


In [2]:
POOL_PATH = os.path.join(study_dir, "study_results_3oh", "few_shot_pool_3oh.json")
STATE_DIR = os.path.join(study_dir, "inference_results_3oh")   # shared with notebook 16
CATALOGUE_CSV = os.path.join(project_root, "Notebooks", "cda_class_list.csv")

# Which unlabelled spectra (HF inference split) to send:
#   "all":             every unlabelled spectrum in the Q_I window (~205k spectra)
#   "catalogue_type3": spectra the catalogue lists only as generic Type "3"
#   "sclk_file":       the SCLKs in SCLK_FILE (one SCLK per line, or a CSV with an 'sclk' column)
INFERENCE_SCOPE = "all"
SCLK_FILE = None
# Only spectra from 2005 on: the earlier ones are mostly interplanetary cruise data (2000-2004), where E-ring
# 3-OH grains are not expected. SCLK at 2005-01-01 00:00 UTC, from the SCLK-UTC fit to Notebooks/cda_class_list.csv.
START_2005_SCLK = 1483230354
SCLK_RANGE = (START_2005_SCLK, None)  # (min, max) SCLK limits; None = no limit
APPLY_QI_WINDOW = True     # same 10 fC - 2 pC window as the study

BATCH_SIZE = 1000
BATCHES_PER_RUN = 10       # 10 x 1,000 = 10,000 spectra per execution of this notebook
SUBMIT_NEW_BATCHES = True  # False = only collect jobs that were submitted earlier
POLL_SECONDS = 60
SUBMIT_RETRIES = 10        # if Vertex AI refuses a submission (e.g. too many jobs at once), wait and retry
RETRY_WAIT_SECONDS = 300
SAVE_CANDIDATE_PLOTS = True

LOCAL_REQUEST_PATH = os.path.join(STATE_DIR, "batch_requests_16b.jsonl")  # one batch at a time, deleted after upload
JOBS_PATH = os.path.join(STATE_DIR, "submitted_jobs_16b.json")
NB16_REQUEST_PATH = os.path.join(STATE_DIR, "batch_requests.jsonl")
processed_path = os.path.join(STATE_DIR, "processed_sclks.json")
predictions_path = os.path.join(STATE_DIR, "predictions_3oh.csv")
plots_dir = os.path.join(STATE_DIR, "candidate_plots")
PREDICTION_COLUMNS = ["sclk", "event_id", "qi_ampl", "catalogue_class", "predicted_label", "explanation", "batch_job", "classified_at"]

configs = Config()
PROJECT_ID = configs.agent_settings.project_id
BUCKET = configs.agent_settings.bucket_name.replace("gs://", "")
os.makedirs(STATE_DIR, exist_ok=True)
print(f"Project: {PROJECT_ID} | bucket: {BUCKET}")
print(f"State and results: {STATE_DIR}")

Project: turan-genai-bb | bucket: turansgenaibb
State and results: /workspaces/cda-dust-analyzer/Notebooks/3oh_classification/inference_results_3oh


## Step 2: Load the Few-Shot Pool from Notebook 15

In [3]:
with open(POOL_PATH) as f:
    pool = json.load(f)
if pool["time_base"] != c3.time_base_settings():
    raise ValueError(f"The pool was built with other preprocessing/plot settings:\n{pool['time_base']}\n"
                     f"vs. now:\n{c3.time_base_settings()}\nRerun notebook 15.")
if pool["model"] != configs.agent_settings.model:
    print(f"Note: config model is {configs.agent_settings.model}; using the pool's model {pool['model']} (the one that was tested).")

exemplar_sclks = {ex["sclk"] for ex in pool["exemplars"]}
label_counts = pd.Series([ex["label"] for ex in pool["exemplars"]]).value_counts().to_dict()
print(f"Few-shot pool from {pool['created']}: {len(pool['exemplars'])} exemplars {label_counts}, model {pool['model']}")

Few-shot pool from 2026-10-19T14:08:32: 21 exemplars {'non-3-OH': 12, '3-OH': 9}, model gemini-3.5-flash


## Step 3: Load Progress

This step stops if notebook 16 appears to be running. It then loads the processed SCLKs (from notebooks 16 and 16b) and the jobs submitted earlier by this notebook that have not been collected yet.

In [4]:
if os.path.exists(NB16_REQUEST_PATH):
    raise RuntimeError(
        f"{NB16_REQUEST_PATH} exists, so notebook 16 is probably running a batch. Do not run 16 and 16b at the same time.\n"
        "If notebook 16 is stopped and did not submit that batch, delete the file and rerun this cell.")


def save_json_atomic(path, obj):
    """Write to a temporary file first, so an interruption never leaves a half-written state file."""
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(obj, f)
    os.replace(tmp_path, path)


processed = set()
if os.path.exists(processed_path):
    with open(processed_path) as f:
        processed = set(json.load(f))

jobs = []
if os.path.exists(JOBS_PATH):
    with open(JOBS_PATH) as f:
        jobs = json.load(f)
pending_jobs = [j for j in jobs if j["status"] == "submitted"]
in_flight = {s for j in pending_jobs for s in j["sclks"]}

print(f"Already processed: {len(processed):,} spectra.")
print(f"Jobs submitted earlier and not yet collected: {len(pending_jobs)} ({len(in_flight):,} spectra); they are collected in Step 7.")

Already processed: 14,892 spectra.
Jobs submitted earlier and not yet collected: 0 (0 spectra); they are collected in Step 7.


## Step 4: Select the Next Unlabelled Spectra

In [5]:
inf_path = c3.hf_path(c3.HF_INF_FILE)
catalogue = c3.load_catalogue(CATALOGUE_CSV)

meta_all = (c3.read_metadata(inf_path)
            .merge(catalogue[["sclk", "catalogue_class"]], on="sclk", how="left")
            .drop_duplicates("sclk"))
print(f"Loaded metadata of {len(meta_all)} unlabelled spectra.")
meta = meta_all[meta_all["spectrum_length"] == c3.SPECTRUM_LENGTH]
if APPLY_QI_WINDOW:
    meta = c3.in_qi_window(meta)
    print(f"{len(meta)} in the Q_I window (10 fC - 2 pC).")

if INFERENCE_SCOPE == "catalogue_type3":
    meta = meta[meta["catalogue_class"] == "3"]
elif INFERENCE_SCOPE == "sclk_file":
    if not SCLK_FILE:
        raise ValueError("Set SCLK_FILE for INFERENCE_SCOPE = 'sclk_file'.")
    if SCLK_FILE.endswith(".csv"):
        listed = pd.read_csv(SCLK_FILE)["sclk"]
    else:
        with open(SCLK_FILE) as f:
            listed = pd.Series([line.strip() for line in f if line.strip()])
    meta = meta[meta["sclk"].isin(pd.to_numeric(listed, errors="coerce").dropna().astype("int64"))]
elif INFERENCE_SCOPE != "all":
    raise ValueError(f"Unknown INFERENCE_SCOPE: {INFERENCE_SCOPE}")

sclk_min, sclk_max = SCLK_RANGE
if sclk_min is not None:
    meta = meta[meta["sclk"] >= sclk_min]
if sclk_max is not None:
    meta = meta[meta["sclk"] <= sclk_max]
meta = meta[~meta["sclk"].isin(exemplar_sclks)].sort_values("sclk").reset_index(drop=True)

queue = meta[~meta["sclk"].isin(processed | in_flight)].reset_index(drop=True)
run_queue = queue.head(BATCHES_PER_RUN * BATCH_SIZE) if SUBMIT_NEW_BATCHES else queue.head(0)
n_batches = int(np.ceil(len(run_queue) / BATCH_SIZE))

print(f"Scope '{INFERENCE_SCOPE}': {len(meta):,} spectra; {len(queue):,} not yet processed or submitted.")
if n_batches:
    print(f"This run: {len(run_queue):,} spectra in {n_batches} batch(es), "
          f"SCLK {run_queue['sclk'].min()} - {run_queue['sclk'].max()}; {len(queue) - len(run_queue):,} remain for later runs.")
    request_mb = (sum(len(ex["image_base64"]) for ex in pool["exemplars"]) + 60_000) / 1e6
    print(f"Each local request file: ~{request_mb * min(BATCH_SIZE, len(run_queue)) / 1000:.1f} GB (deleted after upload).")
    n_before_2005 = int((run_queue["sclk"] < START_2005_SCLK).sum())
    if n_before_2005:
        print(f"Note: {n_before_2005:,} of these spectra are from before 2005 (mostly cruise phase). "
              "Set SCLK_RANGE = (START_2005_SCLK, None) to skip them.")
else:
    print("No new batches in this run.")

2026-10-20 07:24:23,039 - [httpx] - INFO - HTTP Request: HEAD https://huggingface.co/datasets/CosmicDustGroup/cassini-cda-spectra/resolve/main/data/lvl2/cda_qm_spectra_pre2008277_inf_lvl2.parquet "HTTP/1.1 302 Found"


Loaded metadata of 756013 unlabelled spectra.
204907 in the Q_I window (10 fC - 2 pC).
Scope 'all': 191,840 spectra; 181,842 not yet processed or submitted.
This run: 10,000 spectra in 10 batch(es), SCLK 1491674813 - 1506226353; 171,842 remain for later runs.
Each local request file: ~1.3 GB (deleted after upload).


## Step 5: Load the Raw Spectra

The spectra of this run's batches are loaded, plus those of jobs still waiting to be collected (needed for the candidate plots). The inference file is streamed in chunks.

In [6]:
needed_sclks = set(run_queue["sclk"]) | in_flight
raw_spectra = c3.read_spectra(inf_path, needed_sclks).drop_duplicates("sclk").set_index("sclk")["spectrum"]
meta_lookup = meta_all.set_index("sclk")
print(f"Loaded {len(raw_spectra)} raw spectra.")

Loaded 10000 raw spectra.


## Step 6: Prepare, Upload and Submit the Batches

One batch at a time: write the local request file, upload it, submit the job, record the job, delete the local file, then move to the next batch. The jobs run on Vertex AI in the meantime.

In [7]:
def submit_with_retries(gcs_source, display_name):
    for attempt in range(1, SUBMIT_RETRIES + 1):
        try:
            return c3.submit_batch_job(PROJECT_ID, pool["model"], gcs_source, f"gs://{BUCKET}/output/3oh", display_name)
        except RuntimeError as e:
            if attempt == SUBMIT_RETRIES:
                raise
            print(f"Submission refused ({e}).\nRetry {attempt}/{SUBMIT_RETRIES - 1} in {RETRY_WAIT_SECONDS} s...")
            time.sleep(RETRY_WAIT_SECONDS)


run_stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
for b in range(n_batches):
    batch_meta = run_queue.iloc[b * BATCH_SIZE:(b + 1) * BATCH_SIZE].copy()
    print(f"\n--- Batch {b + 1}/{n_batches}: {len(batch_meta)} spectra "
          f"(SCLK {batch_meta['sclk'].min()} - {batch_meta['sclk'].max()}) ---")
    batch_meta["spectrum"] = list(raw_spectra.loc[batch_meta["sclk"]])
    batch_df = c3.preprocess(batch_meta)

    c3.write_request_file(LOCAL_REQUEST_PATH, batch_df, pool)
    display_name = f"cda-3oh-inf-{run_stamp}-b{b + 1:02d}"
    gcs_source = c3.upload_jsonl(PROJECT_ID, BUCKET, LOCAL_REQUEST_PATH, f"input/3oh/{display_name}.jsonl")
    job_name = submit_with_retries(gcs_source, display_name)

    jobs.append({
        "job_name": job_name,
        "display_name": display_name,
        "submitted_at": datetime.now().isoformat(timespec="seconds"),
        "sclks": [int(s) for s in batch_df["sclk"]],
        "status": "submitted",
    })
    save_json_atomic(JOBS_PATH, jobs)
    os.remove(LOCAL_REQUEST_PATH)
    print(f"Submitted {display_name}: {job_name}")

n_waiting = sum(j["status"] == "submitted" for j in jobs)
print(f"\nSubmission finished. {n_waiting} job(s) waiting for results.")


--- Batch 1/10: 1000 spectra (SCLK 1491674813 - 1498613846) ---


Rendering requests:   0%|          | 0/1000 [00:00<?, ?it/s]

Wrote 1000 requests (1350 MB) to /workspaces/cda-dust-analyzer/Notebooks/3oh_classification/inference_results_3oh/batch_requests_16b.jsonl


Uploading batch_requests_16b.jsonl:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Submitted cda-3oh-inf-20261020-072432-b01: projects/355771430623/locations/global/batchPredictionJobs/6443215791776071680

--- Batch 2/10: 1000 spectra (SCLK 1498613984 - 1500128804) ---


...

Submitted cda-3oh-inf-20261020-072432-b09: projects/355771430623/locations/global/batchPredictionJobs/8361467758059192320

--- Batch 10/10: 1000 spectra (SCLK 1506186794 - 1506226353) ---


Rendering requests:   0%|          | 0/1000 [00:00<?, ?it/s]

Wrote 1000 requests (1352 MB) to /workspaces/cda-dust-analyzer/Notebooks/3oh_classification/inference_results_3oh/batch_requests_16b.jsonl


Uploading batch_requests_16b.jsonl:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Submitted cda-3oh-inf-20261020-072432-b10: projects/355771430623/locations/global/batchPredictionJobs/2601363834652327936

Submission finished. 10 job(s) waiting for results.


## Step 7: Wait for the Jobs and Collect the Results

Every `POLL_SECONDS` the notebook checks all submitted jobs. As soon as one has finished, its predictions are downloaded and parsed, as in notebook 16, and appended to `predictions_3oh.csv` and `processed_sclks.json`, and plots of the 3-OH candidates are saved. Spectra without a usable answer, and all spectra of failed jobs, are sent again in a later run.

In [8]:
def collect(job, status):
    predictions = pd.DataFrame(list(c3.iter_batch_predictions(PROJECT_ID, BUCKET, status)),
                               columns=["sclk", "predicted_label", "explanation", "error"])
    answered = (predictions[predictions["predicted_label"].isin(c3.BINARY_LABELS)]
                .dropna(subset=["sclk"]).astype({"sclk": "int64"}).drop_duplicates("sclk"))
    answered = answered[answered["sclk"].isin(job["sclks"]) & ~answered["sclk"].isin(processed)]
    answered = (meta_lookup.loc[answered["sclk"], ["event_id", "qi_ampl", "catalogue_class"]].reset_index()
                .merge(answered[["sclk", "predicted_label", "explanation"]], on="sclk"))
    answered["batch_job"] = job["job_name"]
    answered["classified_at"] = datetime.now().isoformat(timespec="seconds")

    answered[PREDICTION_COLUMNS].to_csv(predictions_path, mode="a", header=not os.path.exists(predictions_path), index=False)
    processed.update(int(s) for s in answered["sclk"])
    save_json_atomic(processed_path, sorted(processed))

    candidate_sclks = answered.loc[answered["predicted_label"] == c3.LABEL_3OH, "sclk"]
    if SAVE_CANDIDATE_PLOTS and len(candidate_sclks):
        os.makedirs(plots_dir, exist_ok=True)
        candidate_df = meta_lookup.loc[candidate_sclks, ["spectrum_length"]].reset_index()
        candidate_df["spectrum"] = list(raw_spectra.loc[candidate_df["sclk"]])
        for row in c3.preprocess(candidate_df).itertuples(index=False):
            with open(os.path.join(plots_dir, f"{row.sclk}.png"), "wb") as f:
                f.write(c3.render_spectrum_png(row.spectrum, f"Sample {row.sclk}"))
    return len(answered), len(candidate_sclks)


pending = [j for j in jobs if j["status"] == "submitted"]
print(f"Waiting for {len(pending)} job(s); checking every {POLL_SECONDS} s.")
while pending:
    for job in pending:
        status = c3.get_batch_job(job["job_name"])
        state = status.get("state", "UNKNOWN")
        job["last_state"] = state
        if state in c3.FINISHED_STATES:
            n_answered, n_3oh = collect(job, status)
            job.update(status="collected", final_state=state, collected_at=datetime.now().isoformat(timespec="seconds"),
                       answered=n_answered, candidates_3oh=n_3oh)
            save_json_atomic(JOBS_PATH, jobs)
            print(f"{job['display_name']}: {state} -> {n_answered} answered, {n_3oh} labelled 3-OH, "
                  f"{len(job['sclks']) - n_answered} without a usable answer (sent again in a later run).")
        elif state in c3.FAILED_STATES:
            job.update(status="failed", final_state=state)
            save_json_atomic(JOBS_PATH, jobs)
            print(f"{job['display_name']}: {state}; its spectra are sent again in a later run. "
                  f"Details: {status.get('error', 'none')}")
    pending = [j for j in jobs if j["status"] == "submitted"]
    if pending:
        states = dict(Counter(j["last_state"].replace("JOB_STATE_", "") for j in pending))
        print(f"[{datetime.now():%H:%M:%S}] {len(pending)} job(s) not finished: {states}")
        time.sleep(POLL_SECONDS)

print("\n" + "=" * 50 + "\nAll jobs of this run collected.\n" + "=" * 50)

Waiting for 10 job(s); checking every 60 s.
[08:14:03] 10 job(s) not finished: {'RUNNING': 8, 'QUEUED': 1, 'PENDING': 1}
[08:15:20] 10 job(s) not finished: {'RUNNING': 9, 'PENDING': 1}
[08:16:40] 10 job(s) not finished: {'RUNNING': 9, 'PENDING': 1}
[08:18:01] 10 job(s) not finished: {'RUNNING': 9, 'PENDING': 1}
[08:19:39] 10 job(s) not finished: {'RUNNING': 10}


cda-3oh-inf-20261020-072432-b01: JOB_STATE_SUCCEEDED -> 964 answered, 31 labelled 3-OH, 36 without a usable answer (sent again in a later run).
[08:22:21] 9 job(s) not finished: {'RUNNING': 9}
[08:23:35] 9 job(s) not finished: {'RUNNING': 9}


cda-3oh-inf-20261020-072432-b03: JOB_STATE_SUCCEEDED -> 1000 answered, 6 labelled 3-OH, 0 without a usable answer (sent again in a later run).
[08:26:11] 8 job(s) not finished: {'RUNNING': 8}


cda-3oh-inf-20261020-072432-b02: JOB_STATE_SUCCEEDED -> 986 answered, 17 labelled 3-OH, 14 without a usable answer (sent again in a later run).


cda-3oh-inf-20261020-072432-b05: JOB_STATE_SUCCEEDED -> 998 answered, 19 labelled 3-OH, 2 without a usable answer (sent again in a later run).


...

cda-3oh-inf-20261020-072432-b09: JOB_STATE_SUCCEEDED -> 1000 answered, 0 labelled 3-OH, 0 without a usable answer (sent again in a later run).
[08:36:41] 4 job(s) not finished: {'RUNNING': 4}


cda-3oh-inf-20261020-072432-b04: JOB_STATE_SUCCEEDED -> 994 answered, 15 labelled 3-OH, 6 without a usable answer (sent again in a later run).
[08:39:13] 3 job(s) not finished: {'RUNNING': 3}
[08:40:18] 3 job(s) not finished: {'RUNNING': 3}
[08:41:23] 3 job(s) not finished: {'RUNNING': 3}
[08:42:27] 3 job(s) not finished: {'RUNNING': 3}
[08:43:32] 3 job(s) not finished: {'RUNNING': 3}
[08:44:36] 3 job(s) not finished: {'RUNNING': 3}


cda-3oh-inf-20261020-072432-b06: JOB_STATE_SUCCEEDED -> 998 answered, 23 labelled 3-OH, 2 without a usable answer (sent again in a later run).


cda-3oh-inf-20261020-072432-b08: JOB_STATE_SUCCEEDED -> 999 answered, 26 labelled 3-OH, 1 without a usable answer (sent again in a later run).
[08:49:53] 1 job(s) not finished: {'RUNNING': 1}
[08:50:54] 1 job(s) not finished: {'RUNNING': 1}
[08:51:55] 1 job(s) not finished: {'RUNNING': 1}
[08:52:57] 1 job(s) not finished: {'RUNNING': 1}
[08:53:59] 1 job(s) not finished: {'RUNNING': 1}
[08:55:00] 1 job(s) not finished: {'RUNNING': 1}
[08:56:02] 1 job(s) not finished: {'RUNNING': 1}
[08:57:04] 1 job(s) not finished: {'RUNNING': 1}
[08:58:05] 1 job(s) not finished: {'RUNNING': 1}
[08:59:07] 1 job(s) not finished: {'RUNNING': 1}
[09:00:09] 1 job(s) not finished: {'RUNNING': 1}
[09:01:10] 1 job(s) not finished: {'RUNNING': 1}
[09:02:12] 1 job(s) not finished: {'RUNNING': 1}
[09:03:14] 1 job(s) not finished: {'RUNNING': 1}
[09:04:15] 1 job(s) not finished: {'RUNNING': 1}


cda-3oh-inf-20261020-072432-b10: JOB_STATE_SUCCEEDED -> 999 answered, 7 labelled 3-OH, 1 without a usable answer (sent again in a later run).

All jobs of this run collected.


## Step 8: Summary and 3-OH Candidates

In [9]:
if os.path.exists(predictions_path):
    all_predictions = pd.read_csv(predictions_path, dtype={"catalogue_class": str}).drop_duplicates("sclk", keep="last")
    print(f"{len(all_predictions)} spectra classified so far (notebooks 16 and 16b):")
    print(all_predictions["predicted_label"].value_counts().to_string())

    candidates = all_predictions[all_predictions["predicted_label"] == c3.LABEL_3OH].sort_values("sclk")
    candidates.to_csv(os.path.join(STATE_DIR, "candidates_3oh.csv"), index=False)
    print(f"\n{len(candidates)} 3-OH candidates written to {os.path.join(STATE_DIR, 'candidates_3oh.csv')}")
    display(candidates.head(20))
else:
    print("No predictions yet.")

The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.
24829 spectra classified so far (notebooks 16 and 16b):
predicted_label
non-3-OH    24594
3-OH          235

235 3-OH candidates written to /workspaces/cda-dust-analyzer/Notebooks/3oh_classification/inference_results_3oh/candidates_3oh.csv


,sclk,event_id,qi_ampl,catalogue_class,predicted_label,explanation,batch_job,classified_at
5226,1484572784,850,2.330420e-14,NaN,3-OH,"The dominant Na+ peak is located at ~1.31 µs, ...",projects/355771430623/locations/global/batchPr...,2026-10-19T20:46:51
5054,1484579592,7191,2.386450e-14,NaN,3-OH,The spectrum displays a dominant Na+ peak at ~...,projects/355771430623/locations/global/batchPr...,2026-10-19T20:46:51
5047,1487171652,1635,1.685830e-14,NaN,3-OH,"The highest peak is Na+ at ~1.45 µs, and the d...",projects/355771430623/locations/global/batchPr...,2026-10-19T20:46:51
6267,1487344855,7282,1.914300e-14,NaN,3-OH,"The highest peak is Na+ at ~1.75 µs, followed ...",projects/355771430623/locations/global/batchPr...,2026-10-19T20:48:36
6083,1487346465,8079,5.825850e-14,NaN,3-OH,The spectrum displays a dominant Na+ peak at 1...,projects/355771430623/locations/global/batchPr...,2026-10-19T20:48:36
6657,1487357074,11471,3.531050e-14,NaN,3-OH,"The highest peak is Na+ at ~1.90 µs, followed ...",projects/355771430623/locations/global/batchPr...,2026-10-19T20:48:36
5896,1487379956,13851,1.604090e-14,NaN,3-OH,The spectrum displays a dominant Na+ peak at a...,projects/355771430623/locations/global/batchPr...,2026-10-19T20:48:36
6889,1489011316,20005,3.089610e-14,3,3-OH,The spectrum displays a dominant Na+ peak at ~...,projects/355771430623/locations/global/batchPr...,2026-10-19T20:48:36
6439,1489016374,23575,5.953580e-14,3,3-OH,The spectrum displays a dominant Na+ peak at ~...,projects/355771430623/locations/global/batchPr...,2026-10-19T20:48:36
8762,1489083496,50194,1.575300e-14,3,3-OH,The spectrum displays a dominant Na+ peak at a...,projects/355771430623/locations/global/batchPr...,2026-10-19T20:52:20
